In [0]:
# ============================================================
# Ajay | Notebook 4: Silver → Gold Reporting Table
# 4 analytics aggregations for flight dashboard
# ============================================================

dbutils.widgets.text("catalog", "dev_team")
dbutils.widgets.text("schema",  "testing")
CATALOG       = dbutils.widgets.get("catalog")
SCHEMA        = dbutils.widgets.get("schema")
SILVER_TABLE = f"`{CATALOG}`.`{SCHEMA}`.Ajay_silver_flight_positions"
GOLD_TABLE   = f"`{CATALOG}`.`{SCHEMA}`.Ajay_gold_flight_summary"

print(f"✅ Config loaded")
print(f"   Silver : {SILVER_TABLE}")
print(f"   Gold   : {GOLD_TABLE}")

✅ Config loaded
   Silver : `dev_team`.`testing`.Ajay_silver_flight_positions
   Gold   : `dev_team`.`testing`.Ajay_gold_flight_summary


In [0]:
from pyspark.sql import functions as F

# Read Silver table
print("📖 Reading Silver table...")
silver_df = spark.table(SILVER_TABLE)

total = silver_df.count()
print(f"✅ Silver table loaded!")
print(f"   ✈️  Total aircraft : {total}")

📖 Reading Silver table...
✅ Silver table loaded!
   ✈️  Total aircraft : 10676


In [0]:
# Metric 1: Total flights by origin country
print("📊 Building Metric 1 — Flights by Country...")

flights_by_country = silver_df \
    .groupBy("origin_country") \
    .agg(F.count("icao24").alias("count_value")) \
    .orderBy(F.desc("count_value")) \
    .withColumn("metric_name",  F.lit("flights_by_country")) \
    .withColumn("metric_group", F.col("origin_country")) \
    .withColumn("double_value", F.lit(None).cast("double")) \
    .withColumn("bool_value",   F.lit(None).cast("boolean"))

print(f"✅ Metric 1 ready!")
print(f"\n🌍 Top 10 Countries by Flight Count:")
flights_by_country.select(
    "metric_group", "count_value"
).show(10, truncate=30)

📊 Building Metric 1 — Flights by Country...
✅ Metric 1 ready!

🌍 Top 10 Countries by Flight Count:
+--------------+-----------+
|  metric_group|count_value|
+--------------+-----------+
| United States|       6397|
|        Canada|        435|
|United Kingdom|        390|
|       Ireland|        332|
|       Germany|        252|
|        Turkey|        193|
|         Spain|        172|
|        France|        171|
|         Malta|        168|
|         China|        130|
+--------------+-----------+
only showing top 10 rows


In [0]:
# Metric 2: Flights on ground vs airborne
print("📊 Building Metric 2 — Ground vs Air...")

ground_vs_air = silver_df \
    .groupBy("on_ground") \
    .agg(F.count("icao24").alias("count_value")) \
    .withColumn(
        "metric_group",
        F.when(F.col("on_ground") == True, "On Ground").otherwise("In Air")
    ) \
    .withColumn("metric_name",  F.lit("ground_vs_air")) \
    .withColumn("double_value", F.lit(None).cast("double")) \
    .withColumn("bool_value",   F.col("on_ground"))

print(f"✅ Metric 2 ready!")
print(f"\n✈️  Ground vs Air Breakdown:")
ground_vs_air.select(
    "metric_group", "count_value", "bool_value"
).show()

📊 Building Metric 2 — Ground vs Air...
✅ Metric 2 ready!

✈️  Ground vs Air Breakdown:
+------------+-----------+----------+
|metric_group|count_value|bool_value|
+------------+-----------+----------+
|      In Air|       9790|     false|
|   On Ground|        886|      true|
+------------+-----------+----------+



In [0]:
# Metric 3: Average baro altitude by country (airborne only)
print("📊 Building Metric 3 — Avg Altitude by Country...")

avg_alt_by_country = silver_df \
    .filter(F.col("on_ground") == False) \
    .filter(F.col("baro_altitude").isNotNull()) \
    .groupBy("origin_country") \
    .agg(
        F.round(F.avg("baro_altitude"), 2).alias("double_value"),
        F.count("icao24").alias("count_value")
    ) \
    .orderBy(F.desc("double_value")) \
    .withColumn("metric_name",  F.lit("avg_altitude_by_country")) \
    .withColumn("metric_group", F.col("origin_country")) \
    .withColumn("bool_value",   F.lit(None).cast("boolean"))

print(f"✅ Metric 3 ready!")
print(f"\n🏔️  Top 10 Countries by Avg Altitude (metres):")
avg_alt_by_country.select(
    "metric_group", "double_value", "count_value"
).show(10, truncate=30)

📊 Building Metric 3 — Avg Altitude by Country...
✅ Metric 3 ready!

🏔️  Top 10 Countries by Avg Altitude (metres):
+------------------------------+------------+-----------+
|                  metric_group|double_value|count_value|
+------------------------------+------------+-----------+
|      Islamic Republic of Iran|     12496.8|          1|
|            Dominican Republic|     12192.0|          1|
|Democratic Republic of the ...|     11582.4|          1|
|           Republic of Moldova|     11582.4|          4|
|                       Ecuador|    11292.84|          1|
|                    Seychelles|     11277.6|          1|
|                    Cape Verde|     11277.6|          1|
|                     Sri Lanka|    11129.01|          2|
|                       Morocco|    11080.39|         25|
|                      Pakistan|     11010.9|          1|
+------------------------------+------------+-----------+
only showing top 10 rows


In [0]:
# Metric 4: Top 10 countries by active (airborne) aircraft
print("📊 Building Metric 4 — Top 10 Active Countries...")

top10_countries = silver_df \
    .filter(F.col("on_ground") == False) \
    .groupBy("origin_country") \
    .agg(F.countDistinct("icao24").alias("count_value")) \
    .orderBy(F.desc("count_value")) \
    .limit(10) \
    .withColumn("metric_name",  F.lit("top10_active_aircraft")) \
    .withColumn("metric_group", F.col("origin_country")) \
    .withColumn("double_value", F.lit(None).cast("double")) \
    .withColumn("bool_value",   F.lit(None).cast("boolean"))

print(f"✅ Metric 4 ready!")
print(f"\n🏆 Top 10 Countries by Active Airborne Aircraft:")
top10_countries.select(
    "metric_group", "count_value"
).show(10, truncate=30)

📊 Building Metric 4 — Top 10 Active Countries...
✅ Metric 4 ready!

🏆 Top 10 Countries by Active Airborne Aircraft:
+--------------+-----------+
|  metric_group|count_value|
+--------------+-----------+
| United States|       5910|
|        Canada|        378|
|United Kingdom|        361|
|       Ireland|        309|
|       Germany|        228|
|        Turkey|        182|
|         Malta|        162|
|         Spain|        155|
|        France|        148|
|        Brazil|        123|
+--------------+-----------+



In [0]:
# Combine all 4 metrics into one Gold summary table
print("🔗 Combining all 4 metrics...")

gold_df = flights_by_country \
    .select("metric_name", "metric_group", "count_value", "double_value", "bool_value") \
    .union(
        ground_vs_air.select("metric_name", "metric_group", "count_value", "double_value", "bool_value")
    ) \
    .union(
        avg_alt_by_country.select("metric_name", "metric_group", "count_value", "double_value", "bool_value")
    ) \
    .union(
        top10_countries.select("metric_name", "metric_group", "count_value", "double_value", "bool_value")
    ) \
    .withColumn("insert_dttm", F.current_timestamp()) \
    .withColumn("update_dttm", F.current_timestamp()) \
    .withColumn("inserted_by", F.current_user()) \
    .withColumn("updated_by",  F.current_user())

# Write Gold table
print(f"💾 Writing to Gold table: {GOLD_TABLE}")

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_TABLE)

gold_count = spark.table(GOLD_TABLE).count()

print(f"✅ Gold table written successfully!")
print(f"   📋 Table     : {GOLD_TABLE}")
print(f"   📊 Row count : {gold_count}")

🔗 Combining all 4 metrics...
💾 Writing to Gold table: `dev_team`.`testing`.Ajay_gold_flight_summary
✅ Gold table written successfully!
   📋 Table     : `dev_team`.`testing`.Ajay_gold_flight_summary
   📊 Row count : 216


In [0]:
# Preview all 4 metrics in Gold table
print("🏆 GOLD TABLE — All Metrics Summary:\n")

for metric in ["flights_by_country", "ground_vs_air", 
               "avg_altitude_by_country", "top10_active_aircraft"]:
    print(f"{'='*55}")
    print(f"📊 {metric}")
    print(f"{'='*55}")
    spark.table(GOLD_TABLE) \
        .filter(F.col("metric_name") == metric) \
        .select("metric_group", "count_value", "double_value") \
        .orderBy(F.desc("count_value")) \
        .show(5, truncate=30)

print(f"\n✅ All metrics verified!")
print(f"   📋 Table     : {GOLD_TABLE}")
print(f"   📊 Row count : {spark.table(GOLD_TABLE).count()}")
print(f"\n🏁 Notebook 4 Complete!")
print(f"🎉 Full Medallion Pipeline DONE!")

🏆 GOLD TABLE — All Metrics Summary:

📊 flights_by_country
+--------------+-----------+------------+
|  metric_group|count_value|double_value|
+--------------+-----------+------------+
| United States|       6397|        NULL|
|        Canada|        435|        NULL|
|United Kingdom|        390|        NULL|
|       Ireland|        332|        NULL|
|       Germany|        252|        NULL|
+--------------+-----------+------------+
only showing top 5 rows
📊 ground_vs_air
+------------+-----------+------------+
|metric_group|count_value|double_value|
+------------+-----------+------------+
|      In Air|       9790|        NULL|
|   On Ground|        886|        NULL|
+------------+-----------+------------+

📊 avg_altitude_by_country
+--------------+-----------+------------+
|  metric_group|count_value|double_value|
+--------------+-----------+------------+
| United States|       5868|     5599.25|
|        Canada|        377|     6768.12|
|United Kingdom|        357|     8256.94|
|    